In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

Carregando base de dados

In [2]:
repositorio = '../data/raw/Obesity.csv'
base = pd.read_csv(repositorio)

In [3]:
base.head()

,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [4]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Gender          2111 non-null   object 
 1   Age             2111 non-null   float64
 2   Height          2111 non-null   float64
 3   Weight          2111 non-null   float64
 4   family_history  2111 non-null   object 
 5   FAVC            2111 non-null   object 
 6   FCVC            2111 non-null   float64
 7   NCP             2111 non-null   float64
 8   CAEC            2111 non-null   object 
 9   SMOKE           2111 non-null   object 
 10  CH2O            2111 non-null   float64
 11  SCC             2111 non-null   object 
 12  FAF             2111 non-null   float64
 13  TUE             2111 non-null   float64
 14  CALC            2111 non-null   object 
 15  MTRANS          2111 non-null   object 
 16  Obesity         2111 non-null   object 
dtypes: float64(8), object(9)
memory u

Criando a Feature IMC

In [5]:
base['IMC'] = base['Weight'] / (base['Height'] ** 2)

Feature Engineering, Label Encoding

In [6]:
mapa_binario = {'no': 0, 'yes': 1}
mapa_genero = {'Male': 0, 'Female': 1}
mapa_scc = {'no': 1, 'yes': 0}
mapa_frequencia = {
    'no': 0,
    'Sometimes': 1,
    'Frequently': 2,
    'Always': 3
}

In [7]:
colunas_remover = []

In [8]:
for col in ['family_history', 'FAVC', 'SMOKE']:
    base[col + '_Encoding'] = base[col].map(mapa_binario)
    colunas_remover.append(col)

In [9]:
base['Gender_Encoding'] = base['Gender'].map(mapa_genero)
colunas_remover.append('Gender')

In [10]:
base['SCC_Encoding'] = base['SCC'].map(mapa_scc)
colunas_remover.append('SCC')

In [11]:
for col in ['CAEC', 'CALC']:
    base[col +'_Encoding'] = base[col].map(mapa_frequencia)
    colunas_remover.append(col)

In [12]:
base = base.drop(colunas_remover, axis=1)

In [13]:
print('\n--- Reultado Codificação ---')
print(base.filter(regex='_Encoding').head())
print(f'\nTotal de Colunas: {base.shape[1]}')


--- Reultado Codificação ---
   family_history_Encoding  FAVC_Encoding  SMOKE_Encoding  Gender_Encoding  \
0                        1              0               0                1   
1                        1              0               1                1   
2                        1              0               0                0   
3                        0              0               0                0   
4                        0              0               0                0   

   SCC_Encoding  CAEC_Encoding  CALC_Encoding  
0             1              1              0  
1             0              1              1  
2             1              1              2  
3             1              1              2  
4             1              1              1  

Total de Colunas: 18


Feature Engineering, One-Hot Encoding

In [14]:
df_dummies = pd.get_dummies(base['MTRANS'], prefix='MTRANS')

base = pd.concat([base, df_dummies], axis=1)
base = base.drop('MTRANS', axis=1)


In [15]:
print('\n--- Meios de Transporte ---')
print(base.filter(regex='MTRANS_').head())


--- Meios de Transporte ---
   MTRANS_Automobile  MTRANS_Bike  MTRANS_Motorbike  \
0              False        False             False   
1              False        False             False   
2              False        False             False   
3              False        False             False   
4              False        False             False   

   MTRANS_Public_Transportation  MTRANS_Walking  
0                          True           False  
1                          True           False  
2                          True           False  
3                         False            True  
4                          True           False  


Label Encoding Feature Target

In [16]:
mapa_obesidade = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}

In [17]:
base['Obesity_Encoding'] = base['Obesity'].map(mapa_obesidade)
base = base.drop('Obesity', axis=1)

In [18]:
print('\n--- Nível de Obesidade ---')
print(base['Obesity_Encoding'].head())
print(base['Obesity_Encoding'].value_counts())


--- Nível de Obesidade ---
0    1
1    1
2    1
3    2
4    3
Name: Obesity_Encoding, dtype: int64
Obesity_Encoding
4    351
6    324
5    297
2    290
3    290
1    287
0    272
Name: count, dtype: int64


In [19]:
base.isnull().sum()

Age                             0
Height                          0
Weight                          0
FCVC                            0
NCP                             0
CH2O                            0
FAF                             0
TUE                             0
IMC                             0
family_history_Encoding         0
FAVC_Encoding                   0
SMOKE_Encoding                  0
Gender_Encoding                 0
SCC_Encoding                    0
CAEC_Encoding                   0
CALC_Encoding                   0
MTRANS_Automobile               0
MTRANS_Bike                     0
MTRANS_Motorbike                0
MTRANS_Public_Transportation    0
MTRANS_Walking                  0
Obesity_Encoding                0
dtype: int64

Treinamento do Modelo de Machine Learning

In [20]:
x = base.drop('Obesity_Encoding', axis=1)
y = base['Obesity_Encoding']

In [21]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [22]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, Y_train)

RandomForestClassifier(random_state=42)

In [23]:
print('Modelo Treinado')
print(f'Quantidade de Dados para Treino: {X_train.shape[0]}')
print(f'Quantidade de Dados para o Teste: {X_test.shape[0]}')

Modelo Treinado
Quantidade de Dados para Treino: 1688
Quantidade de Dados para o Teste: 423


Medindo a precisão do Modelo

In [24]:
Y_pred = model.predict(X_test)

In [25]:
accuracy = accuracy_score(Y_test, Y_pred)
print('--- Avaliação do Modelo ---')
print(f'Acurácia Geral do Modelo: {accuracy:.4f}')

--- Avaliação do Modelo ---
Acurácia Geral do Modelo: 0.9882


In [26]:
conf_matrix = confusion_matrix(Y_test, Y_pred)
print('\nMatriz de Confusão:\n', conf_matrix)


Matriz de Confusão:
 [[53  1  0  0  0  0  0]
 [ 0 58  0  0  0  0  0]
 [ 0  1 56  1  0  0  0]
 [ 0  0  0 58  0  0  0]
 [ 0  0  0  0 70  0  0]
 [ 0  0  0  0  0 59  1]
 [ 0  0  0  0  0  1 64]]


In [27]:
print('\nRelatório de Classificação:\n', classification_report(Y_test, Y_pred))


Relatório de Classificação:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99        54
           1       0.97      1.00      0.98        58
           2       1.00      0.97      0.98        58
           3       0.98      1.00      0.99        58
           4       1.00      1.00      1.00        70
           5       0.98      0.98      0.98        60
           6       0.98      0.98      0.98        65

    accuracy                           0.99       423
   macro avg       0.99      0.99      0.99       423
weighted avg       0.99      0.99      0.99       423



Salvando o modelo

In [28]:
modelo_caminho = '../models/random_forest_obesity_model.pkl'

In [29]:
joblib.dump(model, modelo_caminho)
print(f'Modelo salvo em: {modelo_caminho}')

Modelo salvo em: ../models/random_forest_obesity_model.pkl
